<div class="alert alert-info">
    <b>Imports</b>
</div>

In [1]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

import pandas as pd

<div class="alert alert-warning">
    <b>Pfad festlegen</b><br>
    - Pfad zu einer zu verwenden Datenbank festlegen (als .csv)
</div>

In [2]:
# Hauptordner
base_dir = Path.cwd()

# Pfad zur Datenbank
db_root = (
    base_dir.parent
    / "1.2_Rock-Type-Mapping"
    / "Rock-Type_Mapping"
)

<div class="alert alert-info">
    <b>Zeitstempel-Ordner</b><br>
    - Falls die Dateien in Zeitstempel Ordner gespeichert worden sind, kann folgender Code verwendet werden
</div>

In [3]:
if not db_root.exists():
    raise FileNotFoundError(f"Pfad existiert nicht: {db_root}")

# Ordner mit neustem Zeitstempel finden
timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise RuntimeError("Keine Ordner mit Zeitstemepln gefunden")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print("Neuester Ordner:", latest_folder.name)

# CSV-Datei darin finden - es sollte nur eine vorhanden sein
csv_files = list(latest_folder.glob("*.csv"))
if not csv_files:
    raise RuntimeError(f"Keine CSV-Datei gefunden: {latest_folder}")

database_path = csv_files[0]
print("Verwendete Datenbank:", database_path.name)

# Datenbank laden
df = pd.read_csv(database_path, low_memory=False)
print("Datenbank geladen. Zeilen:", len(df))
df.head()

Neuester Ordner: 2025-11-27_12-22-40
Verwendete Datenbank: finalized_database_with_lithology.csv
Datenbank geladen. Zeilen: 173091


,location,well_or_spring_name,WGS84_latitude,WGS84_longitude,depth_bgl_in_m,rock_type,stratigraphic_period,temperature_in_c,electrical_conductivity_25c_in_uS/cm,pH,...,Sr_in_umol/L,NH4_in_umol/L,Fe_in_mmol/L,Mn_in_mmol/L,F_in_umol/L,NO3_in_mmol/L,H2SiO3_in_umol/L,HS_in_mmol/L,Database_number,Database_name
0,Euganei (Euganean geothermal area - Battaglia ...,Fonte Le Bibite,45.284444,11.778856,NaN,unconsolidated sediments,NaN,48.9,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
1,Euganei (Euganean geothermal area - Cinto Euga...,Bagno Nord,45.286944,11.649967,NaN,unconsolidated sediments,NaN,34.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
2,Euganei (Euganean geothermal area - Cinto Euga...,Bagno Sud,45.285000,11.648300,NaN,unconsolidated sediments,NaN,30.3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
3,Euganei (Euganean geothermal area - Cinto Euga...,Fattorelle,45.295833,11.645522,NaN,unconsolidated sediments,NaN,21.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
4,Acqui (comment Jacek: general area?),23,44.691677,8.453005,NaN,carbonate sedimentary rocks,NaN,59.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,ALPS_cleaned_Database


<div class="alert alert-info">
    <b>Erste Übersicht der Zielvariable "Rock Type"</b><br>
    - Rock-Type wird ale erste Zielvariable verwendet
    - Diese kann in folgendem betrachtet werden
</div>

In [4]:
print(df.shape) # Zeilen, Spalten
print(df.columns) # Spaltennamen
df.info() # Datentypen
df.describe().T.head(30) # statistische Übersicht für numerische Spalten

(173091, 31)
Index(['location', 'well_or_spring_name', 'WGS84_latitude', 'WGS84_longitude',
       'depth_bgl_in_m', 'rock_type', 'stratigraphic_period',
       'temperature_in_c', 'electrical_conductivity_25c_in_uS/cm', 'pH',
       'redox_potential_in_mV', 'total_dissolved_solids_in_mmol/L',
       'O2_in_mmol/L', 'Na_in_mmol/L', 'Mg_in_mmol/L', 'Ca_in_mmol/L',
       'Cl_in_mmol/L', 'SO4_in_mmol/L', 'HCO3_in_mmol/L', 'Li_in_mmol/L',
       'K_in_mmol/L', 'Sr_in_umol/L', 'NH4_in_umol/L', 'Fe_in_mmol/L',
       'Mn_in_mmol/L', 'F_in_umol/L', 'NO3_in_mmol/L', 'H2SiO3_in_umol/L',
       'HS_in_mmol/L', 'Database_number', 'Database_name'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173091 entries, 0 to 173090
Data columns (total 31 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   location                              141376 non-null  object 
 1   well_or_spri

,count,mean,std,min,25%,50%,75%,max
WGS84_latitude,161147.0,39.001400,6.195224,-40.858890,35.265680,39.370911,42.720460,1.205600e+02
WGS84_longitude,161411.0,-100.906219,26.173494,-176.108333,-112.333030,-105.510270,-98.314130,1.348011e+02
redox_potential_in_mV,562.0,251.253756,1156.690752,-406.000000,-198.350000,-38.500000,190.750000,1.140000e+04
total_dissolved_solids_in_mmol/L,173091.0,1740.486226,2792.266855,0.000000,7.052186,243.195916,2407.735422,2.974370e+04
O2_in_mmol/L,308.0,0.119817,0.128262,0.000000,0.018751,0.090631,0.189231,1.143821e+00
Na_in_mmol/L,125928.0,918.313456,1247.201780,0.000000,26.707532,242.194691,1557.214429,1.889549e+04
Mg_in_mmol/L,132974.0,32.335192,63.529789,0.000000,0.740588,5.760132,41.966674,5.641226e+03
Ca_in_mmol/L,137810.0,106.109000,217.733855,0.000000,1.522032,14.396926,107.334448,4.824567e+03
Cl_in_mmol/L,146669.0,1111.977793,1596.055775,0.000000,6.657264,215.119887,1813.822285,2.747532e+04
SO4_in_mmol/L,125607.0,9.453873,18.727021,0.000000,0.426817,2.186134,10.942848,1.919634e+03


In [5]:
# Testen ob Zielvariable "rock_type" existiert
if "rock_type" not in df.columns:
    raise KeyError("'rock_type' fehlt in der Datenbank")

# Überprüfen
df["rock_type"].value_counts(dropna=False).head(20)

rock_type
unconsolidated sediments           71827
siliciclastic sedimentary rocks    48163
NaN                                13759
mixed sedimentary rocks             9088
basic volcanic rocks                7122
carbonate sedimentary rocks         7105
acid plutonic rocks                 6277
acid volcanic rocks                 3940
water bodies                        3888
intermediate volcanic rocks          806
metamorphic rocks                    646
evaporites                           253
pyroclastic rocks                    121
intermediate plutonic rocks           81
basic plutonic rocks                   8
ice and glaciers                       4
no data                                3
Name: count, dtype: int64

<div class="alert alert-info">
    <b>Preprocessing</b><br>
    - Feature Festlegung - Dies kann basierend auf der vorhergehenden explorativen Datenanalyse geschehen
</div>

In [6]:
feature_cols = [
    "HCO3_in_mmol/L",
    "SO4_in_mmol/L",
    "pH",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "total_dissolved_solids_in_mmol/L"
]

In [7]:
target_col = "rock_type"

# Prüfen, welche Spalten passen
existing_features = [c for c in feature_cols if c in df.columns]
missing_features = [c for c in feature_cols if c not in df.columns]

print("Vorhandene Feature-Spalten:", existing_features)
print("Nicht vorhandene Feature-Spalten:", missing_features)

# Nur wenn gültiger Rock Type
df_valid = df.dropna(subset=[target_col]).copy()

# Fehlende Werte pro Feature
missing_counts = df_valid[existing_features].isna().sum()
print("\nFehlende Werte pro Feature:\n", missing_counts)

# Zählen aller Zeilen mit vollständigen Features:
df_full = df_valid.dropna(subset=existing_features)

print("\nGesamtzeilen mit rock_type:", len(df_valid))
print("Zeilen mit vollständigen Features:", len(df_full))
print("Verlust an Zeilen:", len(df_valid) - len(df_full))

# Optional: erste Zeilen ausgeben
df_full.head()

Vorhandene Feature-Spalten: ['HCO3_in_mmol/L', 'SO4_in_mmol/L', 'pH', 'Na_in_mmol/L', 'Mg_in_mmol/L', 'Ca_in_mmol/L', 'Cl_in_mmol/L', 'total_dissolved_solids_in_mmol/L']
Nicht vorhandene Feature-Spalten: []

Fehlende Werte pro Feature:
 HCO3_in_mmol/L                      62698
SO4_in_mmol/L                       45052
pH                                  41013
Na_in_mmol/L                        44913
Mg_in_mmol/L                        38967
Ca_in_mmol/L                        34427
Cl_in_mmol/L                        25863
total_dissolved_solids_in_mmol/L        0
dtype: int64

Gesamtzeilen mit rock_type: 159332
Zeilen mit vollständigen Features: 61110
Verlust an Zeilen: 98222


,location,well_or_spring_name,WGS84_latitude,WGS84_longitude,depth_bgl_in_m,rock_type,stratigraphic_period,temperature_in_c,electrical_conductivity_25c_in_uS/cm,pH,...,Sr_in_umol/L,NH4_in_umol/L,Fe_in_mmol/L,Mn_in_mmol/L,F_in_umol/L,NO3_in_mmol/L,H2SiO3_in_umol/L,HS_in_mmol/L,Database_number,Database_name
18,Craveggia (Comano) - Piemont,Craveggia,46.196800,8.541600,NaN,metamorphic rocks,NaN,25.7,3.61,9.34,...,NaN,NaN,NaN,NaN,136.856511,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
30,Masino - Lombardie,Masino,46.243520,9.725340,NaN,basic plutonic rocks,NaN,37.9,5.91,8.65,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
31,Pre-St. Didier (San Desiderio) - Aoste Valley,Pre-Saint-Didier,45.761890,6.968900,NaN,carbonate sedimentary rocks,NaN,22.3,8.0,7.6,...,NaN,NaN,NaN,NaN,15.264765,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
36,Simplon,Q7,46.262740,8.109190,1650.0,unconsolidated sediments,NaN,38.6,15.11,7.5,...,NaN,NaN,NaN,NaN,15.264765,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
38,Valdieri,Terme II-2,44.206389,7.269967,NaN,metamorphic rocks,NaN,63.0,NaN,9.19,...,NaN,NaN,NaN,NaN,626.381724,NaN,NaN,NaN,5.0,ALPS_cleaned_Database


In [8]:
import pandas as pd

target_col = "rock_type"

# Prüfen, welche Spalten passen
df_valid = df.dropna(subset=[target_col]).copy()

# Alle Feature-Spalten in numerisch wandeln
for col in feature_cols:
    df_valid[col] = pd.to_numeric(df_valid[col], errors="coerce")

# Zeilen behalten in denen alle Features enthalten sind
df_full = df_valid.dropna(subset=feature_cols)

print("Gesamtzeilen mit rock_type:", len(df_valid))
print("Zeilen mit vollständigen (numerischen) Features:", len(df_full))
print("Verlust an Zeilen:", len(df_valid) - len(df_full))

df_full.head()


Gesamtzeilen mit rock_type: 159332
Zeilen mit vollständigen (numerischen) Features: 61109
Verlust an Zeilen: 98223


,location,well_or_spring_name,WGS84_latitude,WGS84_longitude,depth_bgl_in_m,rock_type,stratigraphic_period,temperature_in_c,electrical_conductivity_25c_in_uS/cm,pH,...,Sr_in_umol/L,NH4_in_umol/L,Fe_in_mmol/L,Mn_in_mmol/L,F_in_umol/L,NO3_in_mmol/L,H2SiO3_in_umol/L,HS_in_mmol/L,Database_number,Database_name
18,Craveggia (Comano) - Piemont,Craveggia,46.196800,8.541600,NaN,metamorphic rocks,NaN,25.7,3.61,9.34,...,NaN,NaN,NaN,NaN,136.856511,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
30,Masino - Lombardie,Masino,46.243520,9.725340,NaN,basic plutonic rocks,NaN,37.9,5.91,8.65,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
31,Pre-St. Didier (San Desiderio) - Aoste Valley,Pre-Saint-Didier,45.761890,6.968900,NaN,carbonate sedimentary rocks,NaN,22.3,8.0,7.60,...,NaN,NaN,NaN,NaN,15.264765,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
36,Simplon,Q7,46.262740,8.109190,1650.0,unconsolidated sediments,NaN,38.6,15.11,7.50,...,NaN,NaN,NaN,NaN,15.264765,NaN,NaN,NaN,5.0,ALPS_cleaned_Database
38,Valdieri,Terme II-2,44.206389,7.269967,NaN,metamorphic rocks,NaN,63.0,NaN,9.19,...,NaN,NaN,NaN,NaN,626.381724,NaN,NaN,NaN,5.0,ALPS_cleaned_Database


In [9]:
X = df_full[feature_cols]
y = df_full[target_col]

# Labels encoden
encoder = LabelEncoder()
y_enc = encoder.fit_transform(y)

# Train/Test-Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Pipeline: Skalierung + SVC
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=10, gamma="scale"))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(
    y_test, y_pred,
    target_names=encoder.classes_
))


                                 precision    recall  f1-score   support

            acid plutonic rocks       0.00      0.00      0.00       159
            acid volcanic rocks       0.00      0.00      0.00        52
           basic plutonic rocks       0.00      0.00      0.00         1
           basic volcanic rocks       0.00      0.00      0.00        73
    carbonate sedimentary rocks       0.00      0.00      0.00       242
                     evaporites       0.00      0.00      0.00        32
    intermediate plutonic rocks       0.00      0.00      0.00        12
    intermediate volcanic rocks       0.00      0.00      0.00        44
              metamorphic rocks       0.00      0.00      0.00        71
        mixed sedimentary rocks       0.50      0.01      0.02       598
              pyroclastic rocks       0.00      0.00      0.00         2
siliciclastic sedimentary rocks       0.60      0.62      0.61      4365
       unconsolidated sediments       0.66      0.

C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [10]:
mapping = {
    "siliciclastic sedimentary rocks": "SEDIMENTARY",
    "mixed sedimentary rocks": "SEDIMENTARY",
    "carbonate sedimentary rocks": "SEDIMENTARY",
    "evaporites": "SEDIMENTARY",
    "unconsolidated sediments": "SEDIMENTARY",

    "acid plutonic rocks": "MAGMATIC",
    "intermediate plutonic rocks": "MAGMATIC",
    "basic plutonic rocks": "MAGMATIC",
    "acid volcanic rocks": "MAGMATIC",
    "intermediate volcanic rocks": "MAGMATIC",
    "basic volcanic rocks": "MAGMATIC",
    "pyroclastic rocks": "MAGMATIC",

    "metamorphic rocks": "METAMORPHIC",

    "water bodies": "WATER"
}

df["rock_type_simplified"] = df["rock_type"].map(mapping)


In [11]:
df_simple = df.dropna(subset=["rock_type_simplified"]).copy()


In [12]:
target_col = "rock_type_simplified"


In [13]:
X = df_full[feature_cols]
y = df_full[target_col]

# Labels encoden
encoder = LabelEncoder()
y_enc = encoder.fit_transform(y)

# Train/Test-Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Pipeline: Skalierung + SVC
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=10, gamma="scale"))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(
    y_test, y_pred,
    target_names=encoder.classes_
))


KeyError: 'rock_type_simplified'